# Kavra stüdyosu (Colab, GPU)

Kavra'nın **tamamını** (arayüz + video render + XTTS v2/Anka/Piper, istersen Chatterbox) Colab GPU'sunda kurar
ve cloudflared ile internete açar. Bilgisayarında hiçbir şey çalışmasına gerek yok.

1. **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU** (veya daha iyisi).
2. Hücreleri sırayla çalıştır. İlk kurulum ~10-15 dk sürer; aynı oturumda tekrar çalıştırınca atlanır.
3. 2. hücrenin yazdırdığı **bağlantıyı** aç: arayüz şifreyle korunur, bağlantı şifreyi içerir — kimseyle paylaşma.
4. Videonu üret, indir; bitince 4. hücreyle kapat (GPU kotan boşa gitmez).

**İsteğe bağlı (sol menü → Secrets 🔑, not defterine erişim ver):**
- `KAVRA_ERISIM_SIFRESI` — her oturumda aynı kalan şifre (en az 16 karakter). Yoksa her seferinde rastgele üretilir.
- `GEMINI_API_KEY`, `OPENAI_API_KEY`, `ELEVENLABS_API_KEY` — anlatı/seslendirme anahtarları (arayüzden de girilebilir).

> Ses ayarlarında *Çalıştırma yeri* **Bu bilgisayar** kalmalı: burada "bu bilgisayar" Colab'ın GPU'su.
> `DRIVE_KAYDET` açık değilse projeler oturum kapanınca silinir — videonu kapatmadan önce indir.

In [ ]:
# 1) Depoyu GitHub'dan çek ve kur
REPO_URL = "https://github.com/davutsimsekk/Kavra.git"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
CHATTERBOX_KUR = False  # @param {type:"boolean"}

import os, subprocess, sys

REPO = "/content/Kavra"
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "YOK - Çalışma zamanı türünü GPU yap (CPU'da da çalışır ama XTTS çok yavaş olur)")

if os.path.isdir(f"{REPO}/.git"):
    subprocess.run(["git", "-C", REPO, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO], check=True)
print(subprocess.run(["git", "-C", REPO, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

sys.path.insert(0, REPO)
from tools import colab_studio
colab_studio.install(chatterbox=CHATTERBOX_KUR)

In [ ]:
# 2) Kavra'yı başlat ve internete aç
DRIVE_KAYDET = False  # @param {type:"boolean"}

studio = colab_studio.start(use_drive=DRIVE_KAYDET)

In [ ]:
# 3) Açık tut: sunucuyu ve tüneli izler, düşerse yeniden başlatır (■ ile izlemeyi bırakabilirsin)
studio.watch()

In [ ]:
# 4) Bitti mi? Önce 3. hücreyi ■ ile durdur, sonra bunu çalıştır: Kavra kapanır, GPU serbest kalır.
# Drive'a kaydetmediysen indirmediğin videolar silinir!
CALISMA_ZAMANINI_KAPAT = True  # @param {type:"boolean"}

studio.stop()
if CALISMA_ZAMANINI_KAPAT:
    from google.colab import runtime
    runtime.unassign()